config

In [ ]:
!pip install supabase pandas requests

2

In [ ]:
import google.auth
import google.auth.transport.requests
from google.oauth2 import service_account
import requests as req_lib
import json

FIREBASE_PROJECT_ID = "ini"  # ganti dengan project ID Firebase kamu
SERVICE_ACCOUNT_FILE = "ini"  # file yang didownload dari Firebase

def get_fcm_access_token() -> str:
    """Ambil OAuth2 access token dari service account untuk FCM v1 API."""
    credentials = service_account.Credentials.from_service_account_file(
        SERVICE_ACCOUNT_FILE,
        scopes=["https://www.googleapis.com/auth/firebase.messaging"]
    )
    request = google.auth.transport.requests.Request()
    credentials.refresh(request)
    return credentials.token

def kirim_fcm_ke_semua(title: str, body: str, is_urgent: bool = False):
    """
    Kirim notif FCM ke semua token yang tersimpan di tabel fcm_tokens.
    Dipanggil dari ETL setelah detect gempa baru.
    """
    # Ambil semua token dari Supabase
    res = supabase.table("fcm_tokens").select("token").execute()
    tokens = [row["token"] for row in res.data if row.get("token")]

    if not tokens:
        print("  ⚠️  Tidak ada FCM token tersimpan")
        return

    access_token = get_fcm_access_token()
    url = f"https://fcm.googleapis.com/v1/projects/{FIREBASE_PROJECT_ID}/messages:send"
    headers = {
        "Authorization": f"Bearer {access_token}",
        "Content-Type": "application/json",
    }

    berhasil = 0
    gagal = 0

    for token in tokens:
        payload = {
            "message": {
                "token": token,
                "notification": {
                    "title": title,
                    "body": body,
                },
                "data": {
                    "is_urgent": "true" if is_urgent else "false",
                },
                "android": {
                    "priority": "high",  # WAJIB biar notif langsung muncul
                    "notification": {
                        "channel_id": "ews_urgent" if is_urgent else "ews_info",
                        "sound": "default" if is_urgent else None,
                        "default_vibrate_timings": is_urgent,
                    }
                }
            }
        }

        try:
            resp = req_lib.post(url, headers=headers, json=payload, timeout=10)
            if resp.status_code == 200:
                berhasil += 1
            else:
                gagal += 1
                print(f"  ❌ FCM error token {token[:20]}...: {resp.text}")
        except Exception as e:
            gagal += 1
            print(f"  ❌ FCM exception: {e}")

    print(f"  📲 FCM terkirim: {berhasil} sukses, {gagal} gagal dari {len(tokens)} token")


In [ ]:

import time
import requests
import pandas as pd
from datetime import datetime, timedelta
from supabase import create_client, Client

3

In [ ]:
SUPABASE_URL  = "ini juga"
SUPABASE_KEY  = "ganti pakek key kamu"
INTERVAL_DETIK = 30

# Satu client untuk semua schema
supabase: Client = create_client(SUPABASE_URL, SUPABASE_KEY)

# Helper: akses tabel di schema gold
# Kompatibel semua versi supabase-py tanpa ClientOptions
def gold(table_name: str):
    """Akses tabel di schema gold via supabase.schema()."""
    return supabase.schema("gold").table(table_name)

4

In [ ]:
def parse_kedalaman(raw: str) -> int:
    """'10 km' → 10  |  '68.3 km' → 68  |  '10' → 10"""
    try:
        return int(float(str(raw).lower().replace("km", "").strip()))
    except Exception:
        return 0


def parse_koordinat(raw: str):
    """'lat,lon' → (float lat, float lon)"""
    try:
        parts = str(raw).split(",")
        return float(parts[0].strip()), float(parts[1].strip())
    except Exception:
        return None, None


def parse_waktu(raw: str) -> datetime | None:
    """
    Coba beberapa format waktu yang datang dari 3 sumber berbeda.
    Kembalikan datetime atau None kalau gagal semua.
    """
    fmts = [
        "%d %b %Y %H:%M:%S",   # BMKG  : "05 Mei 2026 13:44:50 WIB"  ← strip " WIB" dulu
        "%d %B %Y %H:%M:%S",
        "%Y-%m-%d %H:%M:%S",   # USGS-HIST / EMSC
        "%d May %Y %H:%M:%S",
        "%d %b %Y %H:%M:%S WIB",
    ]
    # Bulan Indonesia → Inggris (singkat)
    id_bulan = {
        "Januari": "Jan", "Februari": "Feb", "Maret": "Mar",
        "April":   "Apr", "Mei":      "May", "Juni":  "Jun",
        "Juli":    "Jul", "Agustus":  "Aug", "September": "Sep",
        "Oktober": "Oct", "November": "Nov", "Desember":  "Dec",
    }
    s = str(raw).strip()
    for id_b, en_b in id_bulan.items():
        s = s.replace(id_b, en_b)
    s = s.replace(" WIB", "").strip()

    for fmt in fmts:
        try:
            return datetime.strptime(s, fmt)
        except ValueError:
            continue
    return None



Extract

In [ ]:
def extract_bmkg() -> list[dict]:
    res = requests.get(
        "https://data.bmkg.go.id/DataMKG/TEWS/gempadirasakan.json",
        timeout=10
    ).json()
    rows = []
    for item in res["Infogempa"]["gempa"]:
        rows.append({
            "source":    "BMKG",
            "magnitude": float(item["Magnitude"]),
            "wilayah":   item["Wilayah"],
            "waktu":     f"{item['Tanggal']} {item['Jam']}",
            "kedalaman": item["Kedalaman"],
            "koordinat": item["Coordinates"],
        })
    return rows


def extract_usgs() -> list[dict]:
    res = requests.get(
        "https://earthquake.usgs.gov/earthquakes/feed/v1.0/summary/all_hour.geojson",
        timeout=10
    ).json()
    rows = []
    for item in res["features"]:
        props  = item["properties"]
        coords = item["geometry"]["coordinates"]
        dt     = datetime.fromtimestamp(props["time"] / 1000.0)
        rows.append({
            "source":    "USGS",
            "magnitude": float(props["mag"]) if props["mag"] is not None else 0.0,
            "wilayah":   props["place"] or "",
            "waktu":     dt.strftime("%d %b %Y %H:%M:%S"),
            "kedalaman": f"{coords[2]} km",
            "koordinat": f"{coords[1]},{coords[0]}",
        })
    return rows


def extract_emsc() -> list[dict]:
    res = requests.get(
        "https://www.seismicportal.eu/fdsnws/event/1/query?limit=10&format=json",
        timeout=10
    ).json()
    rows = []
    for item in res["features"]:
        props = item["properties"]
        rows.append({
            "source":    "EMSC",
            "magnitude": float(props["mag"]),
            "wilayah":   props["flynn_region"],
            "waktu":     props["time"].replace("T", " ")[:19],
            "kedalaman": f"{props['depth']} km",
            "koordinat": f"{props['lat']},{props['lon']}",
        })
    return rows


def extract_all() -> list[dict]:
    semua = []
    for fn, nama in [(extract_bmkg, "BMKG"), (extract_usgs, "USGS"), (extract_emsc, "EMSC")]:
        try:
            data = fn()
            semua.extend(data)
            print(f"  ✅ {nama}: {len(data)} baris")
        except Exception as e:
            print(f"  ❌ {nama} error: {e}")
    return semua



6

Transform

In [ ]:
def load_gempa_live(rows: list[dict]):
    """Hapus semua data lama, insert yang baru (fresh window)."""
    supabase.table("gempa_live").delete().neq("id", 0).execute()
    if rows:
        supabase.table("gempa_live").insert(rows).execute()
    print(f"  ✅ gempa_live: {len(rows)} baris di-refresh")

7

In [ ]:
_cache_waktu: dict[str, int] = {}   # tanggal_str → id_waktu

def get_or_create_id_waktu(tanggal: datetime) -> int | None:
    tgl_str = tanggal.strftime("%Y-%m-%d")
    if tgl_str in _cache_waktu:
        return _cache_waktu[tgl_str]

    # Cek apakah sudah ada
    res = (
        gold("dim_waktu")
        .select("id_waktu")
        .eq("tanggal", tgl_str)
        .execute()
    )
    if res.data:
        id_w = res.data[0]["id_waktu"]
    else:
        # Insert baru
        ins = gold("dim_waktu").insert({
            "tanggal": tgl_str,
            "tahun":   tanggal.year,
            "bulan":   tanggal.month,
            "hari":    tanggal.day,
        }).execute()
        id_w = ins.data[0]["id_waktu"]

    _cache_waktu[tgl_str] = id_w
    return id_w

8

In [ ]:
_cache_lokasi: dict[tuple, int] = {}   # (lat_2dp, lon_2dp, wilayah) → id_lokasi

def get_or_create_id_lokasi(lat: float, lon: float, wilayah: str) -> int | None:
    # Bulatkan 2 desimal agar noise float tidak bikin duplikat palsu
    key = (round(lat, 2), round(lon, 2), wilayah.strip())
    if key in _cache_lokasi:
        return _cache_lokasi[key]

    res = (
        gold("dim_lokasi")
        .select("id_lokasi")
        .eq("latitude",  str(round(lat, 8)))
        .eq("longitude", str(round(lon, 8)))
        .eq("wilayah",   wilayah.strip())
        .execute()
    )
    if res.data:
        id_l = res.data[0]["id_lokasi"]
    else:
        ins = gold("dim_lokasi").insert({
            "latitude":  round(lat, 8),
            "longitude": round(lon, 8),
            "wilayah":   wilayah.strip(),
        }).execute()
        id_l = ins.data[0]["id_lokasi"]

    _cache_lokasi[key] = id_l
    return id_l

9

In [ ]:
def sudah_ada_di_fact(id_waktu: int, id_lokasi: int, magnitudo: float, kedalaman: int) -> bool:
    res = (
        gold("fact_gempa")
        .select("id_fakta")
        .eq("id_waktu",  id_waktu)
        .eq("id_lokasi", id_lokasi)
        .eq("magnitudo", str(magnitudo))
        .eq("kedalaman", kedalaman)
        .execute()
    )
    return bool(res.data)

10

In [ ]:
def mag_ke_status(mag: float) -> str:
    if mag < 2.0:
        return "Mikrogempa"
    elif mag < 3.0:
        return "Mikrogempa"
    elif mag < 4.0:
        return "Gempa terasa ringan"
    elif mag < 5.0:
        return "Gempa terasa jelas"
    elif mag < 6.0:
        return "Gempa sedang"
    elif mag < 7.0:
        return "Gempa kuat"
    else:
        return "Gempa besar"


11

In [ ]:
def merge_ke_gold(rows: list[dict]):
    baru = 0
    skip_parse = 0
    skip_dedup = 0

    for row in rows:
        # 1. Parse waktu
        dt = parse_waktu(row["waktu"])
        if dt is None:
            skip_parse += 1
            continue

        # 2. Parse koordinat & kedalaman
        lat, lon = parse_koordinat(row["koordinat"])
        if lat is None or lon is None:
            skip_parse += 1
            continue
        kedalaman = parse_kedalaman(row["kedalaman"])
        magnitudo = round(float(row["magnitude"]), 1)

        # 3. Lookup / insert dimensi
        try:
            id_waktu  = get_or_create_id_waktu(dt)
            id_lokasi = get_or_create_id_lokasi(lat, lon, row["wilayah"])
        except Exception as e:
            print(f"  ⚠️  Gagal dim lookup: {e}")
            continue

        # 4. Cek duplikat
        if sudah_ada_di_fact(id_waktu, id_lokasi, magnitudo, kedalaman):
            skip_dedup += 1
            continue

        # 5. Insert ke fact_gempa
        try:
            gold("fact_gempa").insert({
                "id_waktu":    id_waktu,
                "id_lokasi":   id_lokasi,
                "magnitudo":   str(magnitudo),
                "kedalaman":   kedalaman,
                "status_gempa": mag_ke_status(magnitudo),
            }).execute()
            baru += 1
        except Exception as e:
            print(f"  ⚠️  Gagal insert fact: {e}")

    print(f"  ✅ fact_gempa: +{baru} baru | {skip_dedup} duplikat dilewati | {skip_parse} gagal parse")



Load

In [ ]:
# =====================================================================
# CELL INI HARUS JADI CODE CELL (BUKAN MARKDOWN)
# Ini adalah 2 cell terpisah di notebook kamu
# =====================================================================

# ── CELL A: cek_dan_notif (taruh SEBELUM main) ──────────────────────

_last_notified_ids: set = set()  # in-memory anti-spam, di luar fungsi

# Koordinat user default (Samarinda) — ganti sesuai lokasi server/user
# Nanti bisa diambil dari tabel fcm_tokens kalau mau per-user
USER_LAT = -0.5022
USER_LNG = 117.1536

def hitung_jarak_km(lat1, lng1, lat2, lng2) -> float:
    """Hitung jarak 2 titik koordinat dalam km (Haversine)."""
    from math import radians, sin, cos, sqrt, atan2
    R = 6371.0
    dlat = radians(lat2 - lat1)
    dlng = radians(lng2 - lng1)
    a = sin(dlat/2)**2 + cos(radians(lat1)) * cos(radians(lat2)) * sin(dlng/2)**2
    return R * 2 * atan2(sqrt(a), sqrt(1 - a))

def cek_dan_notif_gempa_baru(rows: list[dict]):
    global _last_notified_ids

    res = supabase.table("gempa_live") \
        .select("id, magnitude, wilayah, kedalaman, waktu, koordinat") \
        .order("id", desc=True) \
        .limit(20) \
        .execute()

    for item in res.data:
        item_key = f"{item.get('wilayah', '')}_{item.get('waktu', '')}"
        if item_key in _last_notified_ids:
            continue
        _last_notified_ids.add(item_key)

        mag = float(item.get("magnitude", 0))
        wilayah = item.get("wilayah", "Unknown")
        kedalaman = item.get("kedalaman", "-")
        koordinat = item.get("koordinat", "")

        # Hitung jarak
        distance_km = 9999.0
        if koordinat:
            try:
                parts = koordinat.split(",")
                qlat, qlng = float(parts[0].strip()), float(parts[1].strip())
                distance_km = hitung_jarak_km(USER_LAT, USER_LNG, qlat, qlng)
            except Exception:
                pass

        is_near = distance_km <= 300.0
        is_dangerous = mag >= 5.0

        # Logic sama persis seperti Flutter EWS
        if is_near and is_dangerous:
            title = f"⚠️ AWAS! GEMPA KUAT DEKAT ANDA"
            body = f"M {mag} terdeteksi {distance_km:.0f} km dari Anda. BERLINDUNG!"
            is_urgent = True
        elif is_near and not is_dangerous:
            title = f"ℹ️ Info: Gempa Terasa"
            body = f"Gempa M {mag} berjarak {distance_km:.0f} km di {wilayah}."
            is_urgent = False
        elif not is_near and is_dangerous:
            title = f"Peringatan Gempa Jauh"
            body = f"Gempa kuat M {mag} terjadi di {wilayah}."
            is_urgent = False
        else:
            # Minor & jauh — skip, tidak dinotif
            continue

        print(f"  📲 Kirim FCM: {title}")
        kirim_fcm_ke_semua(title=title, body=body, is_urgent=is_urgent)

    if len(_last_notified_ids) > 500:
        _last_notified_ids = set(list(_last_notified_ids)[-500:])


# ── CELL B: main loop ────────────────────────────────────────────────

def main():
    print("=" * 50)
    print("  SeismoGuard Pipeline — START")
    print(f"  Interval polling : {INTERVAL_DETIK} detik")
    print("=" * 50)

    siklus = 0
    while True:
        siklus += 1
        sekarang = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        print(f"\n[Siklus #{siklus} | {sekarang}]")

        # EXTRACT
        print("→ EXTRACT dari API...")
        rows = extract_all()

        if rows:
            # LOAD ke staging (gempa_live)
            print("→ LOAD ke public.gempa_live...")
            load_gempa_live(rows)

            # KIRIM FCM ke semua device
            cek_dan_notif_gempa_baru(rows)

            # MERGE ke gold
            print("→ MERGE ke gold.fact_gempa...")
            merge_ke_gold(rows)
        else:
            print("  ⚠️  Tidak ada data yang berhasil ditarik, siklus dilewati.")

        print(f"  ⏳ Menunggu {INTERVAL_DETIK} detik...")
        time.sleep(INTERVAL_DETIK)


if __name__ == "__main__":
    main()

  SeismoGuard Pipeline — START
  Interval polling : 30 detik

[Siklus #1 | 2026-05-30 13:26:54]
→ EXTRACT dari API...
  ✅ BMKG: 15 baris
  ✅ USGS: 10 baris
  ✅ EMSC: 10 baris
→ LOAD ke public.gempa_live...
  ✅ gempa_live: 35 baris di-refresh
→ MERGE ke gold.fact_gempa...
  ✅ fact_gempa: +35 baru | 0 duplikat dilewati | 0 gagal parse
  ⏳ Menunggu 30 detik...

[Siklus #2 | 2026-05-30 13:28:04]
→ EXTRACT dari API...
  ✅ BMKG: 15 baris
  ✅ USGS: 10 baris
  ✅ EMSC: 10 baris
→ LOAD ke public.gempa_live...
  ✅ gempa_live: 35 baris di-refresh
→ MERGE ke gold.fact_gempa...
  ✅ fact_gempa: +0 baru | 35 duplikat dilewati | 0 gagal parse
  ⏳ Menunggu 30 detik...
